# P041 Plant Report Pipeline (Colab runtime)

This notebook **is** the Colab runtime that opencode drives through the
`colab-mcp` bridge. The P041 pipeline runs here, in this order:

| Stage | Notebook function | Source |
|---|---|---|
| 1. Ingest | `ingest_base64_csv` / `ingest_drive` / `simulate` | local files (via MCP) or Drive or simulator |
| 2. Validate | `app.services.ingestion.validate_plant_data` | `backend/app` (uploaded from the local repo) |
| 3. Analytics | `app.services.analytics.analyze_plant` | deterministic Python (no LLM) |
| 4. LLM narrative | `app.services.llm.generate_report` | OpenAI-compatible provider (secrets) or `demo-stub` |
| 5. Report | `save_report` | markdown in `/content/p041/reports/` + base64 for download |

Agent contract (opencode): after connecting via `open_colab_browser_connection`,
execute code cells through the bridge. Start with `run_pipeline(use_stub=True)`
for a keyless demo, then switch to a live provider. See `colab/README.md`.

In [ ]:
# --- 1. Setup ---------------------------------------------------------------
!pip install -q fastapi "uvicorn>=0.30" "httpx>=0.27" python-multipart "pydantic>=2" fastmcp
print("deps installed")

In [ ]:
# --- 2. Locate the P041 backend code ----------------------------------------
# The repo has no git/pip package yet, so `backend/app` is pushed into the
# runtime by the agent through the MCP bridge (or via Drive). Pure Python, no
# compiled deps, so it runs as-is on the Colab VM.
import base64, json, os, pathlib, sys

ROOT = pathlib.Path("/content/p041")
BACKEND = ROOT / "backend"
DATA = ROOT / "data"
REPORTS = ROOT / "reports"

def ensure_backend() -> bool:
    if (BACKEND / "app").is_dir():
        sys.path.insert(0, str(BACKEND))
        return True
    return False

print("backend/app present:", ensure_backend())

In [ ]:
# --- 3. Ingestion -----------------------------------------------------------
# A) Local files via the MCP bridge: agent reads the local CSV, base64-encodes
#    it, and calls this with the payload -> written into the Colab runtime.
def ingest_base64_csv(b64: str, name: str = "upload.csv") -> pathlib.Path:
    DATA.mkdir(parents=True, exist_ok=True)
    p = DATA / name
    p.write_bytes(base64.b64decode(b64))
    print(f"ingested {len(p.read_bytes())} bytes -> {p}")
    return p

# B) Google Drive (no MCP needed for the file itself).
def ingest_drive(rel: str = "plant_data.csv") -> pathlib.Path:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print("drive mount skipped:", exc)
    return pathlib.Path("/content/drive/MyDrive") / rel

# C) Deterministic simulator (keyless demo, same as the local dashboard).
def simulate(machines: int = 4, observations: int = 48, seed: int = 7):
    from app.services.simulator import generate_plant_data
    return generate_plant_data(num_machines=machines,
                               observations_per_machine=observations, seed=seed)

In [ ]:
# --- 4. LLM configuration ---------------------------------------------------
# Live mode: OpenAI-compatible endpoint. Keys come from Colab secrets
# (google.colab.userdata) or env vars set by the agent — never hard-coded.
# Demo mode: `demo-stub` renderer below, clearly labeled, keyless fallback.
def llm_env() -> dict:
    try:
        from google.colab import userdata
        get = lambda k: userdata.get(k) or os.environ.get(k, "")
    except Exception:
        get = lambda k: os.environ.get(k, "")
    env = {
        "LLM_PROVIDER": get("LLM_PROVIDER") or "openai-compatible",
        "LLM_API_KEY": get("LLM_API_KEY"),
        "LLM_MODEL": get("LLM_MODEL") or "qwen2.5:7b-instruct-q4_K_M",
        "LLM_BASE_URL": get("LLM_BASE_URL") or "http://localhost:11434/v1",
        "LLM_TIMEOUT_S": get("LLM_TIMEOUT_S") or "60",
    }
    os.environ.update(env)
    return env

def demo_stub_render(report, report_type: str) -> str:
    p = report.plant
    return (
        "# Plant Performance Report\n\n"
        f"Report type: {report_type}. (DEMO STUB - not a real LLM; "
        "no network used.)\n\n"
        f"Production: {p.total_production} units, efficiency: "
        f"{p.average_efficiency}%, energy: {p.total_energy_kwh} kWh, "
        f"downtime: {p.total_downtime_minutes} min, "
        f"anomalies: {len(report.anomalies)}.\n"
    )

print("LLM config ready; providers:", sorted(set(llm_env().values()) & {"openai-compatible", ""}))

In [ ]:
# --- 5. Pipeline: validate -> analytics -> LLM -> report ---------------------
def run_pipeline(csv_path: str | None = None,
                 profile: str = "generic",
                 machines: int = 4, observations: int = 48, seed: int = 7,
                 report_type: str = "daily",
                 use_stub: bool = False) -> dict:
    """Run the full P041 pipeline and return the GeneratedReport as a dict."""
    from app.services.analytics import analyze_plant
    if csv_path is None:
        obs = simulate(machines, observations, seed)
    else:
        import csv as _csv
        rows = list(_csv.DictReader(open(csv_path, newline="", encoding="utf-8-sig")))
        if profile != "generic":
            from app.services.dataset_profiles import map_upload
            rows, _derived = map_upload(rows, profile)
        from app.services.ingestion import validate_plant_data
        obs = validate_plant_data(rows)
    analytics = analyze_plant(obs)

    if use_stub:
        content, verified, provider, model = (
            demo_stub_render(analytics, report_type), True, "demo-stub", "stub")
    else:
        from app.services.llm import generate_report
        content, verified, provider, model = generate_report(
            analytics, report_type, None)  # settings read from env (llm_env)
    return {
        "report_type": report_type,
        "content": content,
        "numbers_verified": verified,
        "provider": provider,
        "model": model,
        "analytics_summary": analytics.plant.model_dump(mode="json"),
        "anomaly_count": len(analytics.anomalies),
    }

In [ ]:
# --- 6. Save + hand back the report ----------------------------------------
import datetime as _dt

def save_report(report: dict) -> dict:
    REPORTS.mkdir(parents=True, exist_ok=True)
    path = REPORTS / ("plant-report-" + _dt.date.today().isoformat() + ".md")
    path.write_text(report["content"], encoding="utf-8")
    report["path"] = str(path)
    report["content_b64"] = base64.b64encode(
        report["content"].encode()).decode()  # agent writes this back to disk
    print("saved:", path)
    return report

# Quick keyless demo (no backend/app needed for this single call):
#   run_pipeline(use_stub=True) -> save_report(...)

# --- 7. How opencode drives this (examples) --------------------------------
# Keyless demo (no LLM key, no upload):
#   save_report(run_pipeline(use_stub=True))
#
# Upload a local CSV through the bridge (agent base64-encodes it first):
#   p = ingest_base64_csv("<base64>", "ai4i.csv")
#   save_report(run_pipeline(csv_path=str(p), profile="ai4i"))
#
# Live LLM (set Colab secrets LLM_API_KEY/LLM_MODEL/LLM_BASE_URL first,
# e.g. OpenRouter https://openrouter.ai/api/v1 or Groq https://api.groq.com/openai/v1):
#   llm_env(); save_report(run_pipeline(report_type="weekly"))
#
# The agent downloads the report by decoding `content_b64` into the local file
# (e.g. /home/vasa/code/plant-report-ai/backend/data/reports/...).

print("pipeline ready. try: save_report(run_pipeline(use_stub=True))")

In [ ]:
# --- 8. Optional: expose named tools to the bridge -------------------------
# Best effort: if the Colab runtime's MCP surface accepts registered tools,
# this registers them; otherwise the agent just calls the functions above
# through the bridge's code-execution tool.
try:
    from fastmcp import FastMCP
    _mcp = FastMCP("p041-pipeline")

    @_mcp.tool()
    def p041_run_pipeline(csv_path: str | None = None, profile: str = "generic",
                          report_type: str = "daily", use_stub: bool = False) -> dict:
        return save_report(run_pipeline(csv_path=csv_path, profile=profile,
                                        report_type=report_type, use_stub=use_stub))

    @_mcp.tool()
    def p041_ingest(b64: str, name: str = "upload.csv") -> str:
        return str(ingest_base64_csv(b64, name))

    print("named tools registered: p041_run_pipeline, p041_ingest")
    # Uncomment to block and serve (only if the runtime expects a long-lived server):
    # _mcp.run()
except Exception as exc:
    print("tool registration skipped (use function calls instead):", exc)